# Python: Algorithms

## 1. Foundations

### 1.1. General framework
- A problem clearly specifies the valid inputs and the required output.
- An algorithm is a step-by-step method that guarantees the correct output for any valid input.
- A program is the implemented solution: the algorithm realized in code, together with the data structures it uses.

### 1.2. Algorithm analysis
Algorithm analysis is the process of evaluating an algorithm to understand how well it works. It focuses on two main questions: does the algorithm produce the correct output, and how efficiently does it use resources?
- Correctness can be argued using test cases or a mathematical proof. Test cases are fast and practical, but they only check a limited set of inputs and can miss edge cases. A mathematical proof is harder but more general, since it can guarantee correctness for all valid inputs. Common proof techniques include reasoning with recursion (often using induction) or establishing a loop invariant.
- Efficiency is captured by complexity analysis. Time complexity describes how the running time grows as the input size increases, while space complexity describes how much memory the algorithm uses as the input size increases.

#### Big O notation
Time/space complexity is usually described by *worst-case* and *average-case*. Worst-case gives an upper bound on how slow or memory-heavy the algorithm can get on any valid input, while average-case estimates typical performance over a distribution of inputs (when that distribution makes sense). We often summarize growth rates using Big-O and compare common forms. A typical ordering is:

$$O(1) < O(\log n) < O(n) < O(n \log n) < O(n^2) < O(2^n) < O(n!)$$

This comparison helps you predict scalability: moving right gets expensive fast, and exponential/factorial growth becomes infeasible even for moderate $n$. To determine complexity for an algorithm:
- First define the input size $n$ and what counts as a basic operation (e.g., comparisons, swaps, hash lookups).
- Then count how many times key operations run: a single loop is usually $O(n)$, nested loops often give $O(n^2)$, and a loop that halves/doubles the remaining work tends to be $O(\log n)$. Divide-and-conquer patterns often lead to $O(n\log n)$ (like split into halves, do linear work per level). Exponential behavior like $O(2^n)$ typically comes from branching recursion that explores many combinations, while $O(n!)$ shows up when you try all permutations. For recursive algorithms, write a recurrence (e.g., $T(n)=2T(n/2)+O(n)$) and solve it.
- Finally, drop constants and lower-order terms to keep the dominant growth rate, and do the same for memory by tracking extra storage (arrays, recursion stack, auxiliary structures).

### 1.3. Some strategies

#### Recursion vs. iteration
Recursion and iteration are two ways to repeat work:
- Iteration uses loops and updates state variables until a stopping condition is reached.
- Recursion expresses a function calling itself on incremental input until reaching a base case. It's the implemented version of Math induction.

While recursion can feel a bit like a brain teaser at first, it's the most nature way to write algorithms on recursive structures. But iteration is usually more efficient and stable in practice. You can perform *recursion elimination* techniques to convert recursive code to iterative.

:::{admonition} Case: Factorial
:class: tip

To calculate the factorial $n!$ recursively, we must clearly define these:
- The base case (when to stop, what to return): when $n=1$, return $1$.
- Induction logic (how the chain of problems relate): $n!=n\times(n-1)!$ Note that each step must get closer to the base case.

:::

In [2]:
def factorial(n):
    if n == 0 or n == 1:
        return 1
    else:
        return n * factorial(n - 1)

print(factorial(5))

120


#### Reduction
Reduction is a problem-solving strategy where you transform a new problem into another problem that you already know how to solve. After solving the target problem, you map the result back to the original. Reduction is not a data structure or a single algorithm family; it is a meta-technique that helps you reuse existing algorithms and proofs. Common patterns include reducing to sorting (e.g., interval problems), reducing to graph problems (shortest path, MST, max flow), or reducing to a known data structure operation (e.g., connectivity queries reduced to union-find). Reductions are also used to prove hardness by showing that solving your problem would imply solving a known hard problem.

## 2. Data structures
Data structures organize data so algorithms can work efficiently. They’re often grouped into *linear* structures, which store items in a single sequence (such as lists, arrays, stacks, and queues), and *non-linear* structures, which store items in branching or network-like relationships (such as trees, graphs, and sets).

### 2.1. Linear structures

#### Array
The core idea of array (like Python `tuple`) is to store items side-by-side in contiguous memory locations. Because every position is directly addressable, indexing an element and iterating through all elements are fast $O(1)$. But inserting and deleting elements can be costly $O(n)$ since it may require shifting other elements. Arrays are fixed size, so resizing is involves creating a new array and copying over elements, which is also $O(n)$.

That's why dynamic array (like Python `list`) uses doubling capacity to amortize the cost of resizing over multiple insertions. The core idea is to act like it can grow: allocate double capacity than actual used slots, so that insertions are $O(1)$. When running out of space (occasionally happens), create a new array with double size and copy over elements. This way, most insertions are still $O(1)$ amortized, while occasional resizing is $O(n)$.

Arrays are great when you need random access, tight loop over data and operations are mostly read/append.

#### Linked list
The core idea of a linked list is to store items as separate nodes connected by pointers, not in contiguous memory. In a singly linked list, each node points to the next; in a doubly linked list, each node points to both next and previous. Pros and cons:
- Because nodes are not contiguous, random access is slow: getting the i-th element requires traversal from the head, so indexing is $O(n)$. Iteration and searching for a value are also $O(n)$.
- The advantage is insertion/deletion without shifting. If you already have a reference to the position (or you insert/delete at the head), you only change a few pointers, so insertion/deletion can be $O(1)$. Doubly linked lists use more memory but support $O(1)$ deletion of a known node and easy backward movement.

Linked lists are useful when you do frequent insertions/deletions near a current position. For heavy indexing and tight loops, arrays/dynamic arrays are usually better.

#### Abstract types
Stack, queue, and deque are abstract data types: they define allowed operations and the order elements come out, but they do not specify how data is stored. In practice they can be implemented using arrays/dynamic arrays or linked lists.
- A stack is LIFO (last-in, first-out). Its core operations are push, pop, and top, which are typically $O(1)$. Stacks are useful for nested structure and backtracking, such as matching parentheses, undo operations, and depth-first search.
- A queue is FIFO (first-in, first-out). Its core operations are enqueue, dequeue, and front, which are typically $O(1)$. Queues are useful for processing in arrival order, such as breadth-first search, simulations, and task scheduling.
- A deque (double-ended queue) supports pushing and popping at both the front and back, typically $O(1)$. Deques are useful when you need both ends, especially in sliding window algorithms (e.g., maintaining window max/min with a monotonic deque).

### 2.2. Non-linear structures

#### Hash table
A hash table is a data structure that uses a hash function to convert a key into a table position, so lookup/insert/delete are typically $O(1)$ on average. Map and set are abstract interfaces: a map supports key-value operations, and a set supports membership operations on keys. In Python, `dict` is a map and `set` is a set; both are built-in classes implemented using hash-table ideas.

Sometimes different keys land in the same position (collision). Implementations handle this, but too many collisions can degrade performance toward $O(n)$ in the worst case. Similar to dynamic arrays, hash tables occasionally resize when they get too full, so inserts stay $O(1)$ amortized. Use them for fast membership tests, frequency counting, caching, and key-value lookup when you don’t need ordering.

#### Heap
A heap is a data structure designed to quickly keep track of the most important element. It maintains a simple rule: in a min-heap, every parent is smaller than its children (in a max-heap, every parent is larger). This doesn't fully sort the data, but it guarantees the minimum/maximum is always at the top.

A priority queue is the abstract interface: insert items with priorities, and repeatedly remove the item with highest priority (or lowest, depending on convention). A heap is the most common way to implement a priority queue.

Because the heap stays roughly balanced, inserting an item and removing the top element both take $O(\log n)$, while just peeking at the top is $O(1)$. Heaps are great when you repeatedly need the next best candidate, like scheduling tasks by priority, selecting the top-k elements, or running graph algorithms such as Dijkstra's shortest path.

#### Tree
A tree is a data structure for representing hierarchical relationships. It consists of nodes connected by edges, with one root node at the top; every node (except the root) has exactly one parent, and can have zero or more children. Because of this structure, trees are great for modeling things like folder structures, organization charts, and parsed expressions.

Common operations include traversals (visit all nodes) like preorder/inorder/postorder and level-order; these take $O(n)$ because you touch each node once. Many tree algorithms are naturally recursive, but can also be written iteratively using an explicit stack/queue.

A very common special case is the binary search tree, where for each node, all keys in the left subtree are smaller and all keys in the right subtree are larger. This ordering makes search/insert/delete efficient: typically $O(\log n)$ when the tree stays reasonably balanced, but it can degrade to $O(n)$ if the tree becomes a chain.

## 3. Exact methods

:::{mermaid}
:align: center

flowchart TD
  P["Exact method"]
  BF["Brute force"]
  BT["Backtracking"]
  BB["Branch and Bound"]
  DC["Divide and Conquer"]
  DP["Dynamic Programming"]

  P --> BF
  BF --> |feasibility<br>pruning| BT
  BT --> |optimality<br>pruning| BB
  P --> DC
  DC --> |memoization<br>tabulation| DP

:::

### 3.1. Brute force
Brute force is the simplest problem-solving approach: it tries all possible options util you find the answer (or the best answer). While often inefficient, brute force is easy to implement and guarantees finding the optimal solution. It's useful for small input sizes and you don't want to overcomplicate things.

:::{admonition} Case: Two sum
:class: tip

- Input: an array of integers `array` and an integer `target`.
- Output: The indices $i,j$ of the two elements that add up to `target`. If no such pair exists, return `None`.

Implementation using brute force involves a nested loop (two levels) to check all pairs. The time complexity is $O(n^2)$.

:::

In [ ]:
def two_sum(array, target):
    n = len(array)
    for i in range(n):
        for j in range(i + 1, n):
            if array[i] + array[j] == target:
                return i, j

two_sum([4, 3, 8, 2, 7, 11, 15], 9)

(3, 4)

### 3.2. Divide and conquer
Divide and conquer solves a problem by splitting it into smaller subproblems of the same type. It involves three main steps:
- Divide: break the input into smaller parts
- Conquer: solve each part (usually recursively) until the parts are small enough to solve directly
- Combine: merge the sub-results into the final answer

It works best when the subproblems are mostly independent and combining results is efficient.

:::{admonition} Case: Binary search
:class: tip

- Input: a sorted array of integers `array` and an integer `target`.
- Output: The index of `target` in `array`, or `-1` if not found.

The core idea of binary search is to repeatedly discard half of the array which is guaranteed not to contain the target. This is done by comparing the target with the middle element of the current array, which requires the array to be sorted. The time complexity of binary search is $O(\log n)$.

:::

In [ ]:
def binary_search_iterative(array, target):
    left, right = 0, len(array) - 1
    while left <= right:
        mid = (left + right) // 2
        if array[mid] == target:
            return mid
        elif array[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1

In [ ]:
def binary_search_recursive(array, target):
    
    def helper(left, right):
        if left > right:
            return -1
        mid = (left + right) // 2
        if array[mid] == target:
            return mid
        if array[mid] < target:
            return helper(mid + 1, right)
        return helper(left, mid - 1)

    return helper(0, len(array) - 1)

:::{admonition} Case: Exponentiation by squaring
:class: tip

$b^n$ can be computed efficiently using the idea of divide and conquer with the time complexity $O(\log n)$:

$$\begin{aligned}
n=2k &\rightarrow b^n = (b^2)^k \\
n=2k+1 &\rightarrow b^n = b\times (b^2)^k
\end{aligned}$$

Base on the above observation, we can keep updating $n\leftarrow\lfloor n/2\rfloor$, squaring the base each time $b\leftarrow b^2$, and multiplying by $b$ until $n=0$. This process is the same as calculating the binary representation of $n$ from the right to the left. For example, $12_{10}=1100_2\rightarrow b^{12} = b^{8}\times b^{4}$.

:::

In [ ]:
def fast_pow(base: float, power: int) -> float:
    result = 1

    while power > 0:
        if power % 2 == 1:
            result *= base
        print(f'{result = :<5} {power = :<3} {base = :<4}')
        base *= base
        power //= 2

    return result

fast_pow(2, 12)

### 3.3. Dynamic programming
Dynamic programming (DP) is a method for solving complex problems by breaking them down into simpler overlapping subproblems. It is applicable when the problem has optimal substructure (the optimal solution can be constructed from optimal solutions of its subproblems) and overlapping subproblems (the same subproblems are solved multiple times). DP can be implemented in two main ways:
- Top-down with memoization: This approach uses recursion to solve the problem, storing the results of subproblems in a cache (usually a dictionary) to avoid redundant calculations.
- Bottom-up with tabulation: This approach builds a table (usually an array) iteratively, starting from the smallest subproblems and working up to the original problem.

:::{admonition} Case: Fibonacci numbers
:class: tip

Fibonacci sequence is define as:
- $F(0)=0;\;F(1)=1$
- $F(n)=F(n-1)+F(n-2)$

The naive recursive solution recomputes the same values many times (overlapping subproblems) so it becomes very slow as $n$ grows. Dynamic programming fixes this by storing results and reusing them, making it $O(n)$ time.

:::

In [23]:
adds = 0

def fibo_naive(n):
    global adds
    if n <= 1:
        return n
    result = fibo_naive(n - 1) + fibo_naive(n - 2)
    adds += 1
    return result

fibo_naive(7)
adds

20

In [ ]:
adds = 0

def fib_topdown(n, memo=None):
    global adds
    if memo is None:
        memo = {0: 0, 1: 1}
    if n in memo:
        return memo[n]
    memo[n] = fib_topdown(n - 1, memo) + fib_topdown(n - 2, memo)
    adds += 1
    return memo[n]

fib_topdown(7)
adds

6

In [22]:
adds = 0

def fib_bottomup(n):
    global adds
    if n < 2:
        return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
        adds += 1
    return b

fib_bottomup(7)
adds

6

:::{admonition} Case: Edit distance
:class: tip

Input two strings $A=a_1a_2\dots a_N$ and $B=b_1b_2\dots b_M$. You are only allowed to insert, delete or substitue a character in $A$. The goal is to find minimum cost to transform $A$ to $B$.

A 2D dynamic programming table can be used to solve this problem. Let $d[n][m]$ be the minimum cost to transform the first $n$ characters of $A$ into the first $m$ characters of $B$. Each cell is built from smaller prefix problems, so we gradually grow the answer.

- $d[0][0]=0$
- $d[n][0]=n$: delete all $n$ characters
- $d[0][m]=m$: insert all $m$ characters

For $n=1,\dots,N$ and $m=1,\dots,M$, compare the last characters $a_n$ and $b_m$:
- If they are the same, no extra cost: $d[n][m]=d[n-1][m-1]$
- Otherwise, take the cheapest of:
  - delete $a_n$: $d[n-1][m]+1$
  - insert $b_m$: $d[n][m-1]+1$
  - substitute $a_n\to b_m$: $d[n-1][m-1]+1$

Think about traversing from the upper left to the bottom right, each cell picks the cheapest last operation from its three neighbors: up = delete, left = insert, diagonal = match/substitute. The final answer is $d[N][M]$.


:::

In [ ]:
def edit_distance(A, B, return_table=False):
    N, M = len(A), len(B)
    d = [[0] * (M + 1) for _ in range(N + 1)]

    for n in range(N + 1):
        d[n][0] = n
    for m in range(M + 1):
        d[0][m] = m

    for n in range(1, N + 1):
        for m in range(1, M + 1):
            if A[n - 1] == B[m - 1]:
                d[n][m] = d[n - 1][m - 1]
            else:
                d[n][m] = min(d[n - 1][m], d[n][m - 1], d[n - 1][m - 1]) + 1

    return d if return_table else d[N][M]

tab = edit_distance('CODE', 'BADGE', return_table=True)

In [ ]:
def print_table(d):
    w = max(
        len(str(d[-1][0])),
        len(str(d[-1][-1])),
        len(str(d[0][-1]))
    )
    for row in d:
        print(" ".join(f"{x:>{w}}" for x in row))

print_table(tab)

### 3.4. Backtracking
Backtracking is trying choices one by one and once it can't lead to a feasible solution; you undo that choice then try the next option. Intuitively, you can think about solving a maze: when you hit a dead end, you go back to the last junction and try a different path.

:::{admonition} Case: N-Queens
:class: tip

The N-Queens problem challenges placing $N$ chess Queens on an $N\times N$ chess board so that no two Queen attack each other. The for any pair of Queens $i,j$ the constraints can be written as follows:
- Different columns: $x_i\neq x_j$
- Different rows: $y_i\neq y_j$
- Different main diagonals: $x_i-y_i\neq x_j-y_j$
- Different anti-diagonals: $x_i+y_i\neq x_j+y_j$

:::

In [ ]:
def n_queens(n=4):
    cols = set(); diag = set(); anti = set()
    pos = [-1] * n
    result = []

    def backtrack(r):
        if r == n:
            result.append([(i, pos[i]) for i in range(n)])
        
        for c in range(n):
            if c in cols or (r-c) in diag or (r+c) in anti:
                continue

            cols.add(c); diag.add(r-c); anti.add(r+c)
            pos[r] = c
            backtrack(r+1)
            cols.remove(c); diag.remove(r-c); anti.remove(r+c)
            pos[r] = -1
    
    backtrack(0)
    return result

#### Branch and bound
Branch and bound is a backtracking variant for optimization problems. It prunes a branch by proving it cannot be better than the best solution so far.

:::{admonition} Case: 0/1 knapsack
:class: tip

The knapsack problem gives you $N$ items, where the items $n$ has weight $w_n$ and value $v_n$. You need to pick a subset of items with max total value and the total weight is at most $W$.

Backtracking allows you to find all *feasible* subsets by pruning overweighted ones. Branch and bound adds a second, stronger prune: with the remaining capacity, even if the partial choice can't beat the best solution so far, stop.

:::

In [ ]:
def knap_bb(w, v, W):
    n = len(w)
    order = sorted(range(n), key=lambda i: v[i] / w[i], reverse=True)
    best, best_take = 0, [0] * n
    take = [0] * n

    def ub(k, cw, cv):  # fractional upper bound
        cap, val = W - cw, cv
        for t in range(k, n):
            i = order[t]
            if w[i] <= cap:
                cap -= w[i]; val += v[i]
            else:
                val += cap * (v[i] / w[i])
                break
        return val

    def dfs(k, cw, cv):
        nonlocal best, best_take
        if ub(k, cw, cv) <= best: return
        if k == n:
            if cv > best: best, best_take = cv, take.copy()
            return
        i = order[k]
        if cw + w[i] <= W:
            take[i] = 1; dfs(k + 1, cw + w[i], cv + v[i])
        take[i] = 0; dfs(k + 1, cw, cv)

    dfs(0, 0, 0)
    return best, [i for i in range(n) if best_take[i]]

# example:
# print(knap_bb([2,3,4,5], [3,4,5,6], 5))  # (7, [0,1])

## 4. Trade-off methods

### 4.1. Heuristic
Heuristic design methods are practical approaches for hard optimization problems where exact algorithms are too slow. They aim to quickly find a good enough solution, often by exploring the search space using rules of thumb (deterministic) or using randomness to escape local traps (randomized). Some notable heuristic methods are:
- Deterministic: greedy, tabu search
- Randomized: genetic, simulated annealing, ant colony

:::{admonition} Case: Travelling salesman problem
:class: tip

Given a list of cities and the distances between each pair of cities, what is the shortest possible route that visits each city exactly once and returns to the origin city? In words, you need to find a shortest Hamilton cycle in a weighted graph. The best solution so far of [this 225 cities] is 126,643.

:::

[this 225 cities]: https://raw.githubusercontent.com/mastqe/tsplib/master/ts225.tsp

In [26]:
import random
import numpy as np
from urllib.request import urlopen

class BaseTSP:
    def __init__(self, path_or_url: str):
        self.pts, self.distances = self._load_tsplib_euc2d(path_or_url)
        self.n = self.distances.shape[0]

    def _load_tsplib_euc2d(self, path_or_url: str):
        if path_or_url.startswith(("http://", "https://")):
            text = urlopen(path_or_url).read().decode("utf-8", errors="replace")
        else:
            with open(path_or_url, "r", encoding="utf-8") as f:
                text = f.read()

        toks = text.replace("\n", " ").split()
        i = toks.index("NODE_COORD_SECTION") + 1

        coords = {}
        while toks[i] != "EOF":
            node = int(toks[i]); x = float(toks[i+1]); y = float(toks[i+2])
            coords[node] = (x, y)
            i += 3

        n = len(coords)
        pts = np.array([coords[k] for k in range(1, n + 1)], dtype=float)

        dx = pts[:, None, 0] - pts[None, :, 0]
        dy = pts[:, None, 1] - pts[None, :, 1]
        distances = np.rint(np.sqrt(dx * dx + dy * dy)).astype(int)  # TSPLIB EUC_2D
        return pts, distances

    def solve(self):
        tour = list(range(self.n))
        random.shuffle(tour)
        return tour

    def evaluate(self, tour):
        if len(tour) != self.n or len(set(tour)) != self.n:
            raise ValueError("Invalid tour: must be a permutation of 0..n-1.")
        t = np.asarray(tour, dtype=int)
        if t.min() < 0 or t.max() >= self.n:
            raise ValueError("Invalid tour: node id out of range.")
        return int(self.distances[t, np.roll(t, -1)].sum())

In [27]:
tsp = BaseTSP("https://raw.githubusercontent.com/mastqe/tsplib/master/ts225.tsp")
tour = tsp.solve()
score = tsp.evaluate(tour)
print(f"{score:,d}")

1,574,979


#### Greedy
A greedy algorithm constructs a solution step-by-step by repeatedly choosing the option that gives the best immediate improvement to the objective accoording to a simple rule, committing to it, and continuing until a complete solution is formed. It's efficient, but it's only guaranteed optimal globally when you can prove it.

In [28]:
class GreedyTSP(BaseTSP):
    def solve(self, start: int = 0):
        n = self.n
        dist = self.distances

        if not (0 <= start < n):
            raise ValueError("start out of range")

        unvis = set(range(n))
        tour = [start]
        unvis.remove(start)
        cur = start

        while unvis:
            nxt = min(unvis, key=lambda j: dist[cur, j])
            tour.append(nxt)
            unvis.remove(nxt)
            cur = nxt

        return tour

In [30]:
tsp = GreedyTSP("https://raw.githubusercontent.com/mastqe/tsplib/master/ts225.tsp")
tour = tsp.solve()
score = tsp.evaluate(tour)
print(f"{score:,d}")

152,493


#### Genetic
Maintain a population of candidate solutions and iteratively improve it. Each round evaluates solution quality (*fitness*), prefers better solutions when choosing parents, creates new solutions by combining parts of parents (*crossover*), and occasionally applies small changes (*mutation*) to keep diversity and explore new possibilities.

In [31]:
import random
import numpy as np

class GeneticTSP(BaseTSP):
    def __init__(self, path_or_url: str, pop=80, gens=600, elite=6, mut_p=0.6):
        super().__init__(path_or_url)
        self.pop, self.gens, self.elite, self.mut_p = pop, gens, elite, mut_p

    def _ox(self, p1, p2):
        a, b = sorted(random.sample(range(self.n), 2))
        child = [-1] * self.n
        child[a:b] = p1[a:b]
        fill = [x for x in p2 if x not in child]
        j = 0
        for idx in list(range(0, a)) + list(range(b, self.n)):
            child[idx] = fill[j]; j += 1
        return child

    def _mutate(self, t):
        if random.random() < self.mut_p:
            i, k = random.sample(range(self.n), 2)
            t[i], t[k] = t[k], t[i]
        return t

    def solve(self):
        n = self.n
        pop = [GreedyTSP.__mro__[1].solve(self)]  # fallback; will be replaced below if you prefer
        pop = []
        # seed with one greedy tour + random tours
        greedy = GreedyTSP.__new__(GreedyTSP)
        greedy.__dict__ = self.__dict__
        pop.append(GreedyTSP.solve(greedy, 0))
        while len(pop) < self.pop:
            t = list(range(n)); random.shuffle(t); pop.append(t)

        scores = [self.evaluate(t) for t in pop]

        def pick():
            cand = random.sample(range(self.pop), 4)
            cand.sort(key=lambda i: scores[i])
            return pop[cand[0]]

        best_i = int(np.argmin(scores))
        best, bestL = pop[best_i][:], scores[best_i]

        for _ in range(self.gens):
            idx = np.argsort(scores)
            new_pop = [pop[i][:] for i in idx[:self.elite]]

            while len(new_pop) < self.pop:
                c = self._ox(pick(), pick())
                c = self._mutate(c)
                if random.random() < 0.3:  # occasional 2-opt kick
                    i = random.randint(1, n - 3)
                    k = random.randint(i + 1, n - 2)
                    c = c[:i] + c[i:k + 1][::-1] + c[k + 1:]
                new_pop.append(c)

            pop = new_pop
            scores = [self.evaluate(t) for t in pop]
            bi = int(np.argmin(scores))
            if scores[bi] < bestL:
                best, bestL = pop[bi][:], scores[bi]

        return best

In [32]:
tsp = GeneticTSP("https://raw.githubusercontent.com/mastqe/tsplib/master/ts225.tsp")
tour = tsp.solve()
score = tsp.evaluate(tour)
print(f"{score:,d}")

150,194


#### Simulated annealing
Start from one solution and repeatedly try a small modification to get a neighboring solution. Improvements are accepted immediately, while worse moves can still be accepted early on to escape local optima; over time this becomes less likely as a temperature parameter gradually decreases, making the search increasingly selective.

In [33]:
import random, math

class SimulatedAnnealingTSP(BaseTSP):
    def __init__(self, path_or_url: str, iters=200000, T0=20000.0, alpha=0.9999):
        super().__init__(path_or_url)
        self.iters, self.T0, self.alpha = iters, T0, alpha

    def _two_opt(self, t, i, k):
        return t[:i] + t[i:k+1][::-1] + t[k+1:]

    def solve(self, init=None):
        cur = (init[:] if init is not None else GreedyTSP.solve(self, 0))
        curL = self.evaluate(cur)
        best, bestL = cur[:], curL

        T = self.T0
        for _ in range(self.iters):
            i = random.randint(1, self.n - 3)
            k = random.randint(i + 1, self.n - 2)
            nxt = self._two_opt(cur, i, k)
            nxtL = self.evaluate(nxt)
            d = nxtL - curL

            if d <= 0 or random.random() < math.exp(-d / max(T, 1e-12)):
                cur, curL = nxt, nxtL
                if curL < bestL:
                    best, bestL = cur[:], curL

            T *= self.alpha

        return best

In [34]:
tsp = SimulatedAnnealingTSP("https://raw.githubusercontent.com/mastqe/tsplib/master/ts225.tsp")
tour = tsp.solve()
score = tsp.evaluate(tour)
print(f"{score:,d}")

152,493


#### Ant colony
Use many agents that build solutions step-by-step. Their choices are guided by accumulated *pheromone* information from previously good solutions (plus a problem-specific heuristic), and after each iteration pheromone is reinforced on good components and evaporates elsewhere, balancing learning good patterns with continued exploration.

In [35]:
import random
import numpy as np

class AntColonyTSP(BaseTSP):
    def __init__(self, path_or_url: str, ants=25, iters=80, alpha=1.0, beta=3.0, rho=0.5, Q=100.0):
        super().__init__(path_or_url)
        self.ants, self.iters = ants, iters
        self.alpha, self.beta = alpha, beta
        self.rho, self.Q = rho, Q

    def solve(self):
        n, dist = self.n, self.distances
        tau = np.ones((n, n), float)
        eta = 1.0 / (dist + 1e-9)

        best, bestL = None, 10**18

        for _ in range(self.iters):
            tours, lens = [], []

            for __ in range(self.ants):
                start = random.randrange(n)
                unvis = set(range(n)); unvis.remove(start)
                tour = [start]; cur = start

                while unvis:
                    cand = list(unvis)
                    w = np.array([(tau[cur, j] ** self.alpha) * (eta[cur, j] ** self.beta) for j in cand], float)
                    s = w.sum()
                    if s <= 0:
                        nxt = random.choice(cand)
                    else:
                        r = random.random() * s
                        cum = 0.0
                        nxt = cand[-1]
                        for j, wij in zip(cand, w):
                            cum += wij
                            if cum >= r:
                                nxt = j
                                break
                    tour.append(nxt); unvis.remove(nxt); cur = nxt

                L = self.evaluate(tour)
                tours.append(tour); lens.append(L)
                if L < bestL:
                    best, bestL = tour[:], L

            tau *= (1.0 - self.rho)  # evaporation
            for tour, L in zip(tours, lens):  # deposit
                dep = self.Q / float(L)
                for i in range(n):
                    a, b = tour[i], tour[(i + 1) % n]
                    tau[a, b] += dep
                    tau[b, a] += dep

        return best

In [36]:
tsp = AntColonyTSP("https://raw.githubusercontent.com/mastqe/tsplib/master/ts225.tsp")
tour = tsp.solve()
score = tsp.evaluate(tour)
print(f"{score:,d}")

141,686


#### Particle swarm
Keep a swarm of candidate solutions that move through the search space. Each candidate adjusts its movement based on its own best position found so far and the best position found by the group, so the swarm collectively drifts toward promising regions while still spreading out enough to explore alternatives.

In [38]:
import numpy as np
import random

class ParticleSwarmTSP(BaseTSP):
    def __init__(self, path_or_url: str, num=40, iters=300, w=0.7, c1=1.5, c2=1.5, two_opt_tries=30000):
        super().__init__(path_or_url)
        self.num, self.iters = num, iters
        self.w, self.c1, self.c2 = w, c1, c2
        self.two_opt_tries = two_opt_tries

    def _perm(self, x):
        return list(np.argsort(x))

    def _two_opt(self, t, i, k):
        return t[:i] + t[i:k+1][::-1] + t[k+1:]

    def _best_2opt(self, t, tries):
        best = t[:]; bestL = self.evaluate(best)
        for _ in range(tries):
            i = random.randint(1, self.n - 3)
            k = random.randint(i + 1, self.n - 2)
            cand = self._two_opt(best, i, k)
            L = self.evaluate(cand)
            if L < bestL:
                best, bestL = cand, L
        return best

    def solve(self):
        n = self.n
        X = np.random.randn(self.num, n)
        V = np.random.randn(self.num, n) * 0.1

        pbestX = X.copy()
        pbestL = np.array([self.evaluate(self._perm(X[i])) for i in range(self.num)], dtype=float)
        g = int(np.argmin(pbestL))
        gX = pbestX[g].copy()
        gL = float(pbestL[g])

        for _ in range(self.iters):
            r1 = np.random.rand(self.num, n)
            r2 = np.random.rand(self.num, n)
            V = self.w * V + self.c1 * r1 * (pbestX - X) + self.c2 * r2 * (gX - X)
            X = X + V

            for i in range(self.num):
                L = self.evaluate(self._perm(X[i]))
                if L < pbestL[i]:
                    pbestL[i] = L
                    pbestX[i] = X[i].copy()
                    if L < gL:
                        gL = L
                        gX = pbestX[i].copy()

        tour = self._perm(gX)
        return self._best_2opt(tour, self.two_opt_tries)

In [39]:
tsp = ParticleSwarmTSP("https://raw.githubusercontent.com/mastqe/tsplib/master/ts225.tsp")
tour = tsp.solve()
score = tsp.evaluate(tour)
print(f"{score:,d}")

194,330


### 4.2. Other methods

#### Randomized
Randomized algorithms use random choices during execution. Two classic types are:
- Monte Carlo algorithms run in a predictable amount of time but may produce an incorrect answer with small probability. For example, a randomized primality test can quickly report that a number is probably prime, though there is still a tiny chance of error.
- Las Vegas algorithms, in contrast, always produce a correct answer, but their running time depends on the random choices made. For example, randomized quicksort always sorts correctly, but it may run faster or slower depending on which random pivots are selected.

#### Approximation
Approximation algorithms are designed for optimization problems where finding the exact optimum may be too expensive, but a solution close to optimal can be found efficiently. They run in polynomial time and provide a provable guarantee on solution quality, usually expressed as how close the result is to the true optimum.

For example, in the Traveling Salesman Problem (TSP), an approximation algorithm may first connect cities in a simple low-cost structure and then turn that structure into a tour, so the route is built from globally sensible connections rather than trying every possible order; the result may not be the shortest possible tour, but it is guaranteed to stay within a known bound of the optimum.

#### Online
Online algorithms make decisions step by step as input arrives, without knowing the full future input in advance. They must commit to each choice before seeing what comes next, so they are usually evaluated by how well they perform compared with an ideal offline algorithm that knows the entire input beforehand. For example, in the paging (cache replacement) problem, requests arrive one by one, and when the cache is full the algorithm must decide immediately which page to evict, even though future requests are unknown.

## 5. Sorting algorithms
Sorting problems ask us to rearrange a collection of items into a specified order, usually ascending or descending according to a key such as value, name, or date. It is a fundamental problem in algorithms because sorted data is easier to search, process, and organize efficiently.

### 5.1. Basic methods

#### Bubble sort
Repeatedly compare adjacent elements and swap them if they are in the wrong order, so larger elements gradually *bubble* to the end. It is mainly useful for learning the basic idea of sorting and swapping, since it is simple but inefficient in practice. Its time complexity is $O(n^2)$ in the average and worst case, and $O(n)$ in the best case if the array is already sorted and we use an early-stop check.

In [ ]:
def bubble_sort(a):
    n = len(a)
    for i in range(n):
        for j in range(n - 1 - i):
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
    return a

#### Selection sort
Repeatedly scan the unsorted part to find the smallest element, then place it into the next correct position at the front. Its main advantage is simplicity and the fact that it performs only $O(n)$ swaps, which can be useful when swapping is expensive. However, it still needs many comparisons, so its time complexity is $O(n^2)$ in the best, average, and worst cases.

In [ ]:
def selection_sort(a):
    n = len(a)
    for i in range(n):
        m = i
        for j in range(i + 1, n):
            if a[j] < a[m]:
                m = j
        a[i], a[m] = a[m], a[i]
    return a

#### Insertion sort
Build the sorted part one element at a time by taking the next element and inserting it into its proper place among the already sorted elements. It works well for small inputs or nearly sorted data, because each new element usually moves only a short distance. Its time complexity is $O(n^2)$ in the average and worst case, but $O(n)$ in the best case when the input is already sorted.

In [ ]:
def insertion_sort(a):
    for i in range(1, len(a)):
        x = a[i]
        j = i - 1
        while j >= 0 and a[j] > x:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = x
    return a

### 5.2. Core efficient methods

#### Merge sort
Divide the array into two halves, sort each half recursively, and then merge the two sorted halves back together. It is a reliable general-purpose sorting method because its running time is consistently good regardless of the input order, and it is especially useful when a stable sort is needed. Its time complexity is $O(n \log n)$ in the best, average, and worst cases, but it usually requires extra memory of $O(n)$ for merging.

In [ ]:
def merge_sort(a):
    if len(a) <= 1:
        return a
    m = len(a) // 2
    left = merge_sort(a[:m])
    right = merge_sort(a[m:])
    i = j = 0
    out = []
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    return out + left[i:] + right[j:]

#### Quick sort
Choose a pivot, partition the array so that smaller elements go to one side and larger elements go to the other, and then recursively sort the two sides. It is often very fast in practice because the partitioning step is efficient and it usually works well in memory, making it one of the most commonly used sorting methods. Its time complexity is $O(n \log n)$ in the best and average cases, but $O(n^2)$ in the worst case when the partitions become very unbalanced.

In [ ]:
def quick_sort(a):
    if len(a) <= 1:
        return a
    p = a[len(a) // 2]
    left = [x for x in a if x < p]
    mid = [x for x in a if x == p]
    right = [x for x in a if x > p]
    return quick_sort(left) + mid + quick_sort(right)

#### Heap sort
First build a heap from the array, then repeatedly remove the largest element from the heap and place it at the end of the array. It is useful when you want a guaranteed $O(n \log n)$ sorting time without needing extra memory for another array, and it also connects naturally to the heap data structure used in priority queues. Its time complexity is $O(n \log n)$ in the best, average, and worst cases, and it uses $O(1)$ extra space.

In [ ]:
def heap_sort(a):
    import heapq
    h = a[:]
    heapq.heapify(h)
    return [heapq.heappop(h) for _ in range(len(h))]

### 5.3. Non-comparison methods

#### Counting sort
Count how many times each value appears, then use those counts to place each element directly into its correct position. It works well when the input values are integers within a small range, because it avoids comparing elements and can sort very quickly under that condition. Its time complexity is $O(n + k)$, where $k$ is the range of possible values, and it usually requires $O(n + k)$ extra space.

In [ ]:
def counting_sort(a):
    if not a:
        return a
    mn, mx = min(a), max(a)
    count = [0] * (mx - mn + 1)
    for x in a:
        count[x - mn] += 1
    out = []
    for i, c in enumerate(count):
        out.extend([i + mn] * c)
    return out

#### Radix sort
Sort the numbers digit by digit, usually starting from the least significant digit and using a stable sorting method such as counting sort at each step. It is effective when sorting integers or strings with fixed-length representations, because it can outperform comparison-based sorting when the number of digits is limited. Its time complexity is $O(d(n + k))$, where $d$ is the number of digits and $k$ is the range of each digit, and it requires extra space depending on the stable subroutine used.

In [ ]:
def radix_sort(a):
    if not a:
        return a
    exp = 1
    mx = max(a)
    while mx // exp > 0:
        buckets = [[] for _ in range(10)]
        for x in a:
            buckets[(x // exp) % 10].append(x)
        a = [x for b in buckets for x in b]
        exp *= 10
    return a

## Resources
- algorithm-visualizer.org - [Algorithm Visualizer](https://algorithm-visualizer.org/brute-force/pagerank)
- visualgo.net - [VisuAlgo](https://visualgo.net/en/)
- `github.com` - [The Algorithms - Python](https://github.com/TheAlgorithms/Python/tree/master)
- betterexplained.com - [Sorting algorithms](https://betterexplained.com/articles/sorting-algorithms/)